# SVM para depressão (NHIS 2025): com e sem variáveis de ansiedade

Este notebook repete com o **SVM linear (`LinearSVC`)** a análise feita com o Random Forest (`../Random Forest/06_random_forest_depressao_2025.ipynb`), o KNN e o XGBoost: comparar o modelo com e sem variáveis de ansiedade e testar se o resultado depende das variáveis cognitivas e de solidão.

## Hipótese

> É possível prever depressão a partir de dados de saúde física e do contexto de vida, **sem** depender de sintomas de transtornos mentais.

Esperamos desempenho **pior** sem os sintomas mentais. A hipótese se sustenta se o modelo ainda tiver poder preditivo real, bem acima do acaso.

## Cenários

1. **Cenário A (exploratório, com ansiedade):** removemos de $X$ apenas os sintomas de depressão (PHQ) e mantemos as variáveis de ansiedade (GAD-7).
2. **Cenário B (hipótese, sem sintomas mentais):** removemos também as variáveis de ansiedade.

## Roteiro

1. Preparação dos dados e cenários.
2. Função de treino e avaliação.
3. Cenários A e B e comparativo.
4. Ajuste de hiperparâmetros (`GridSearchCV`) no cenário B.
5. Teste de sensibilidade: o resultado depende das variáveis cognitivas e de solidão?
6. Priorizando o recall: rastreio em saúde.
7. O que mais dá para fazer pelo recall no SVM.
8. Interpretação e conclusão.

## Desenho do experimento

- Mesma base, **mesmas linhas** (registros sem `-1` nas variáveis do cenário A), mesmo alvo binário (`0` = sem depressão, `1` = com depressão) e mesmo split treino/teste (80/20 estratificado, `random_state=42`) dos outros modelos. Os resultados são comparáveis. A justificativa do alvo binário está no notebook do Random Forest.
- **O `PESO_AMOSTRAL` não é usado no treino.** O desbalanceamento (cerca de 21% de positivos) é tratado com `class_weight='balanced'`, como no Random Forest. O peso entra apenas no cálculo das métricas de avaliação, e as matrizes de confusão são sempre em contagem de registros.
- **Por que o SVM linear:** o `SVC` com kernel RBF tem custo que cresce rapidamente com o número de linhas (17.667 no treino, com mais de 100 colunas após o One-Hot). O `LinearSVC` treina em segundos, permite ver o peso de cada variável e, como os resultados mostram, já alcança o desempenho dos modelos não lineares. O preço é que ele só separa as classes por um hiperplano, sem capturar interações entre variáveis.
- O SVM não gera probabilidades. Classificamos como "com depressão" quando a distância à fronteira (`decision_function`) é positiva, e o **ROC-AUC** é calculado sobre essa distância. O ROC-AUC não depende do limiar e serve para comparar os modelos, mas **a prioridade prática é o recall**.
- **Por que priorizar o recall:** em saúde, os dois erros não custam o mesmo. Um falso negativo é uma pessoa com depressão que o modelo não identifica e que, portanto, não é encaminhada para avaliação. Um falso positivo é uma pessoa sem depressão que passa por uma avaliação a mais. Em um rastreio, é preferível encontrar mais casos, mesmo com mais falsos positivos. Por isso usamos `class_weight='balanced'` (que favorece a classe com depressão) e, na seção 6, mostramos quanto recall é possível ganhar baixando o limiar e quanto isso custa em falsos positivos. Na seção 7, testamos o que mais dá para fazer pelo recall com o SVM.

# 1. Preparação dos dados e definição dos cenários

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET_ORIGINAL = 'SEVERIDADE_DEPRESSAO_PHQ'
TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
RANDOM_STATE = 42

# Sintomas diretos de depressão (PHQ): removidos nos DOIS cenários
COLUNAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ'
]

# Ansiedade (GAD-7): mantida no cenário A, removida no cenário B
COLUNAS_ANSIEDADE = [
    'HISTORICO_TRANSTORNO_ANSIEDADE', 'MED_ANSIEDADE',
    'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE',
    'FREQ_ANSIEDADE_2SEM', 'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
    'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
    'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM',
    'FREQ_MEDO_2SEM', 'SEVERIDADE_ANSIEDADE'
]

# Carregar a base limpa
df = pd.read_csv(Path('../../01 Preparação dos dados/dados/limpos') / 'adult25_limpo.csv').set_index('ID_DOMICILIO')

# Garante que nenhuma coluna foi digitada errada (o drop abaixo ignoraria erros silenciosamente)
ausentes = [c for c in COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE if c not in df.columns]
assert not ausentes, f'Colunas inexistentes na base: {ausentes}'

# Remover registros indefinidos no alvo (-1 e 5) e binarizar: 0 = sem depressão | 1 = com depressão
df = df[~df[TARGET_ORIGINAL].isin([-1, 5])].copy()
df[TARGET_BIN] = (df[TARGET_ORIGINAL] != 1).astype(int)

CENARIOS = {
    'A: com ansiedade (exploratório)': COLUNAS_DEPRESSAO,
    'B: sem sintomas mentais (hipótese)': COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE,
}
nome_a, nome_b = CENARIOS
BASE_REMOVER = [TARGET_BIN, PESO_COLUNA]

# Mesmas linhas nos dois cenários: registros sem -1 nas variáveis do cenário A (o mais restritivo)
X_cenario_a = df.drop(columns=BASE_REMOVER + CENARIOS[nome_a])
df_modelo = df[(X_cenario_a != -1).all(axis=1).to_numpy()].copy()

# Mesmo split nos dois cenários (por posição).
# Só existem DUAS partições: treino (80%) e teste (20%). Não há conjunto de validação separado:
# a validação é a validação cruzada feita DENTRO do treino (hiperparâmetros e limiar). O teste só mede o resultado.
posicoes = np.arange(len(df_modelo))
pos_tr, pos_te = train_test_split(
    posicoes, test_size=0.2, random_state=RANDOM_STATE, stratify=df_modelo[TARGET_BIN]
)
y_tr, y_te = df_modelo[TARGET_BIN].iloc[pos_tr], df_modelo[TARGET_BIN].iloc[pos_te]
w_te = df_modelo[PESO_COLUNA].iloc[pos_te]
prevalencia = np.average(y_te, weights=w_te)

print(f'Registros no experimento: {len(df_modelo)} (treino: {len(pos_tr)} | teste: {len(pos_te)})')
print(f'Proporção de depressão (ponderada pelo peso amostral): {np.average(df_modelo[TARGET_BIN], weights=df_modelo[PESO_COLUNA]):.1%}')
for nome, remover in CENARIOS.items():
    n_feat = df_modelo.drop(columns=BASE_REMOVER + remover).shape[1]
    print(f'{nome}: {n_feat} variáveis')

Registros no experimento: 22084 (treino: 17667 | teste: 4417)
Proporção de depressão (ponderada pelo peso amostral): 21.2%
A: com ansiedade (exploratório): 58 variáveis
B: sem sintomas mentais (hipótese): 46 variáveis


# 2. Função de treino e avaliação

A mesma função é usada nos dois cenários, então a **única diferença** entre eles é o conjunto de variáveis. A configuração-base é `LinearSVC(C=1, class_weight='balanced')` com `dual=False`. As métricas da classe positiva (com depressão) são ponderadas pelo `PESO_AMOSTRAL`, e a matriz de confusão é em **contagem de registros reais** do teste, sem peso. A importância de cada variável é a soma dos coeficientes em valor absoluto das colunas One-Hot da mesma variável original, normalizada para somar 1. Como as variáveis com muitas categorias (como `IDADE`) somam mais colunas, essa medida **favorece variáveis com muitos níveis**. Use-a como indicativo, não como ranking exato. A função `avaliar` recebe o limiar de decisão (padrão: distância 0 à fronteira), o que permite, na seção 6, priorizar o recall.

In [2]:
def preparar_matrizes(colunas_remover):
    """Pré-processamento: padroniza a variável numérica e aplica One-Hot nas categóricas."""
    X = df_modelo.drop(columns=BASE_REMOVER + colunas_remover)
    X_tr, X_te = X.iloc[pos_tr], X.iloc[pos_te]
    num_cols = [c for c in ['RAZAO_RENDA_POBREZA'] if c in X.columns]
    cat_cols = [c for c in X.columns if c not in num_cols]
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ])
    # Sem vazamento do teste: escalonador e One-Hot aprendem só no treino (fit) e apenas aplicam no teste (transform)
    return X, preprocessor, preprocessor.fit_transform(X_tr), preprocessor.transform(X_te)


def novo_svm(**parametros):
    base = dict(C=1.0, dual=False, class_weight='balanced', max_iter=20000, random_state=RANDOM_STATE)
    base.update(parametros)
    return LinearSVC(**base)


def avaliar(escore, limiar=0.0):
    """escore = distância à fronteira (decision_function). Acima do limiar = previsto com depressão."""
    y_pred = (escore > limiar).astype(int)
    metricas = {
        'ROC-AUC': roc_auc_score(y_te, escore, sample_weight=w_te),
        'Acurácia': accuracy_score(y_te, y_pred, sample_weight=w_te),
        'Precisão (depressão)': precision_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'Recall (depressão)': recall_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'F1 (depressão)': f1_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    }
    return metricas, confusion_matrix(y_te, y_pred)


def importancia_agregada(modelo, preprocessor, X):
    """Soma o |coeficiente| das colunas One-Hot de cada variável original (normalizado para somar 1)."""
    pesos = pd.Series(np.abs(modelo.coef_[0]), index=preprocessor.get_feature_names_out())
    originais = sorted(X.columns, key=len, reverse=True)
    agregada = {}
    for nome_col, peso in pesos.items():
        base = nome_col.split('__', 1)[1]
        dono = next((c for c in originais if base == c or base.startswith(c + '_')), base)
        agregada[dono] = agregada.get(dono, 0) + peso
    agregada = pd.Series(agregada)
    return (agregada / agregada.sum()).sort_values(ascending=False)


def treinar_avaliar(colunas_remover, **parametros):
    X, preprocessor, X_tr_p, X_te_p = preparar_matrizes(colunas_remover)
    modelo = novo_svm(**parametros).fit(X_tr_p, y_tr)
    metricas, matriz = avaliar(modelo.decision_function(X_te_p))
    return {'Nº de variáveis': X.shape[1], **metricas}, matriz, importancia_agregada(modelo, preprocessor, X)

resultados = {}

# 3. Cenários A e B

**Cenário A (exploratório):** mantém as variáveis de ansiedade. Mostra o desempenho quando há informação de saúde mental disponível.

In [3]:
resultados[nome_a] = treinar_avaliar(CENARIOS[nome_a])
metricas_a, matriz_a, imp_a = resultados[nome_a]

print(f'--- {nome_a} ---')
for k, v in metricas_a.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_a)
print('\nTop 10 variáveis mais importantes (|coeficiente| agregado):')
print(imp_a.head(10).round(4).to_string())

--- A: com ansiedade (exploratório) ---
Nº de variáveis: 58
ROC-AUC: 0.9145
Acurácia: 0.8463
Precisão (depressão): 0.6086
Recall (depressão): 0.8110
F1 (depressão): 0.6954

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[2994  488]
 [ 178  757]]

Top 10 variáveis mais importantes (|coeficiente| agregado):
IDADE                              0.2127
ESTADO_CIVIL                       0.0657
SEVERIDADE_ANSIEDADE               0.0580
AUTOAVALIACAO_SAUDE_GERAL          0.0539
FREQ_IRRITABILIDADE_2SEM           0.0410
FREQ_MEDO_2SEM                     0.0389
FREQ_ANSIEDADE_2SEM                0.0384
FREQ_SOLITARIO                     0.0367
ULTIMA_VISITA_MEDICO               0.0352
FREQ_DIFICULDADE_DESCANSAR_2SEM    0.0347


**Cenário B (hipótese):** sem sintomas de depressão **nem** de ansiedade. Sobram variáveis sociodemográficas, de saúde física, de acesso a serviços de saúde e de contexto social.

In [4]:
resultados[nome_b] = treinar_avaliar(CENARIOS[nome_b])
metricas_b, matriz_b, imp_b = resultados[nome_b]

print(f'--- {nome_b} ---')
for k, v in metricas_b.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_b)
print('\nTop 10 variáveis mais importantes (|coeficiente| agregado):')
print(imp_b.head(10).round(4).to_string())

--- B: sem sintomas mentais (hipótese) ---
Nº de variáveis: 46
ROC-AUC: 0.8551
Acurácia: 0.7983
Precisão (depressão): 0.5241
Recall (depressão): 0.7320
F1 (depressão): 0.6109

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[2821  661]
 [ 243  692]]

Top 10 variáveis mais importantes (|coeficiente| agregado):
IDADE                             0.4767
FREQ_SOLITARIO                    0.0648
ESTADO_CIVIL                      0.0570
AUTOAVALIACAO_SAUDE_GERAL         0.0487
CATEGORIA_RAZAO_POBREZA           0.0275
DOR_CRONICA                       0.0269
ESCOLARIDADE_MAXIMA_FAMILIA       0.0249
SEPARACAO_LEGAL_CONJUGAL          0.0225
VISITAS_PRONTO_ATENDIMENTO_12M    0.0209
SEXO_CONJUGE                      0.0209


## Comparativo entre os cenários

A coluna **Variação (B − A)** mostra o que se perde ao retirar a ansiedade. A coluna **Referência (acaso)** mostra o que um modelo sem informação conseguiria: ROC-AUC de 0,5 e precisão igual à prevalência.

In [5]:
tabela = pd.DataFrame({nome: res[0] for nome, res in resultados.items()})
tabela['Variação (B − A)'] = tabela[nome_b] - tabela[nome_a]
tabela['Referência (acaso)'] = [np.nan, 0.5, np.nan, prevalencia, np.nan, np.nan]

print(f'Prevalência de depressão no teste (ponderada): {prevalencia:.1%}\n')
tabela.round(4)

Prevalência de depressão no teste (ponderada): 21.6%



,A: com ansiedade (exploratório),B: sem sintomas mentais (hipótese),Variação (B − A),Referência (acaso)
Nº de variáveis,58.0000,46.0000,-12.0000,NaN
ROC-AUC,0.9145,0.8551,-0.0594,0.5000
Acurácia,0.8463,0.7983,-0.0480,NaN
Precisão (depressão),0.6086,0.5241,-0.0845,0.2163
Recall (depressão),0.8110,0.7320,-0.0789,NaN
F1 (depressão),0.6954,0.6109,-0.0845,NaN


### Resultado (configuração-base)

- O ROC-AUC cai de **0,915** (A) para **0,855** (B), uma perda de 0,059, e o recall de 81,1% para 73,2%.
- No cenário A, 5 das 10 variáveis mais importantes são de ansiedade (`SEVERIDADE_ANSIEDADE`, `FREQ_IRRITABILIDADE_2SEM`, `FREQ_MEDO_2SEM`, `FREQ_ANSIEDADE_2SEM` e `FREQ_DIFICULDADE_DESCANSAR_2SEM`), atrás de `IDADE`, `ESTADO_CIVIL` e autoavaliação de saúde. Isso confirma a comorbidade entre os dois quadros e é o motivo de usar o cenário A só como referência.
- Sem nenhum sintoma de saúde mental, o cenário B fica **0,35 de ROC-AUC acima do acaso**: a precisão é de 52,4%, **2,4 vezes a prevalência** (21,6%). O modelo usa só dados físicos, funcionais e sociais.
- **Sensibilidade (recall = 73,2%):** o modelo encontra **692 dos 935 casos de depressão do teste** e deixa de detectar **243** (26% dos casos reais). `class_weight='balanced'` desloca a fronteira em favor da classe com depressão, o que sustenta esse recall. No cenário A, são 757 casos encontrados e 178 perdidos (19%).
- **Custo em precisão:** o ganho de sensibilidade vem com falsos alarmes. **661 das 3.482 pessoas sem depressão** do teste (19%) são sinalizadas, e cerca de 52 em cada 100 pessoas sinalizadas têm depressão segundo o PHQ. Em um rastreio esse custo é aceitável, e a seção 6 mostra até onde vale aumentar o recall.
- No cenário B, `IDADE` concentra boa parte do peso (0,48, inflado por ter muitas categorias), seguida de solidão, estado civil, autoavaliação de saúde, renda e dor crônica. A seção 5 testa o quanto o resultado depende das variáveis cognitivas e da solidão.

# 4. Ajuste de hiperparâmetros (cenário B)

O único hiperparâmetro relevante do SVM linear é o `C`, que controla a regularização (quanto menor, mais regularizado). Testamos 7 valores de `C` (de 0,001 a 1) e as penalidades `l2` e `l1` (esta zera os coeficientes das variáveis pouco úteis) com `GridSearchCV` (validação cruzada de 3 partes), **só no cenário B**, que testa a hipótese. A seleção usa o ROC-AUC e **só os dados de treino**; o teste não participa da escolha. Otimizamos o ROC-AUC, que avalia o modelo em todos os limiares, e não o recall: otimizar só o recall levaria a um modelo que sinaliza todo mundo (recall de 100% e nenhuma utilidade). A prioridade pelo recall é tratada depois, na escolha do limiar (seção 6). Valores de `C` acima de 1 foram descartados porque fazem a penalidade `l1` demorar a convergir sem ganho.

In [6]:
_, _, X_tr_b, X_te_b = preparar_matrizes(CENARIOS[nome_b])

ESPACO_BUSCA = {
    'C': [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1],
    'penalty': ['l2', 'l1'],
}

# dual=False é exigido pela penalidade L1 e é o indicado quando há mais linhas que colunas
busca = GridSearchCV(
    novo_svm(), ESPACO_BUSCA, scoring='roc_auc', cv=3, n_jobs=-1,
)
# Hiperparâmetros escolhidos só com validação cruzada no treino (cv=3): o teste não participa da busca
busca.fit(X_tr_b, y_tr)
print('Melhores hiperparâmetros:', busca.best_params_)

modelo_aj = busca.best_estimator_
metricas_aj, matriz_aj = avaliar(modelo_aj.decision_function(X_te_b))
print('\nMatriz de confusão do modelo ajustado (nº de registros do teste):')
print(matriz_aj)
comparacao = pd.DataFrame({'B base': {k: resultados[nome_b][0][k] for k in metricas_aj}, 'B ajustado': metricas_aj})
comparacao['Variação'] = comparacao['B ajustado'] - comparacao['B base']
comparacao.round(4)

Melhores hiperparâmetros: {'C': 0.1, 'penalty': 'l1'}

Matriz de confusão do modelo ajustado (nº de registros do teste):
[[2824  658]
 [ 247  688]]


,B base,B ajustado,Variação
ROC-AUC,0.8551,0.8557,0.0005
Acurácia,0.7983,0.7995,0.0012
Precisão (depressão),0.5241,0.5266,0.0025
Recall (depressão),0.7320,0.7237,-0.0084
F1 (depressão),0.6109,0.6096,-0.0013


### Resultado do ajuste

- A busca escolheu `penalty = 'l1'` e `C = 0,1`: um modelo **bem regularizado**, que descarta as variáveis menos úteis.
- O ganho é praticamente nulo: o ROC-AUC vai de 0,8551 para **0,8557** (+0,0005), e precisão, recall e F1 variam menos de 0,01. Mais uma vez, o limite está na **informação disponível nas variáveis**, não nos hiperparâmetros.

# 5. Teste de sensibilidade: o resultado depende das variáveis cognitivas e de solidão?

`TIPO_DIFICULDADE_COGNITIVA` vem da pergunta sobre dificuldade para **lembrar ou se concentrar**. Dificuldade de concentração também é item do PHQ, que já removemos. Há o risco de essas variáveis reintroduzirem sintomas depressivos no modelo. O mesmo cuidado vale para a solidão. Repetimos o cenário B (mesmas linhas, mesmo split e os hiperparâmetros ajustados) retirando progressivamente:

1. `TIPO_DIFICULDADE_COGNITIVA` sozinha;
2. o **bloco cognitivo** inteiro (`TIPO_DIFICULDADE_COGNITIVA`, `QTD_DIFICULDADE_COGNITIVA` e `FREQUENCIA_FALHA_MEMORIA`, que vêm da mesma pergunta);
3. o bloco cognitivo **e** `FREQ_SOLITARIO`.

Retirar só uma variável é um teste fraco, porque as outras do bloco carregam a mesma informação. Por isso o passo 2 é o mais informativo.

In [7]:
BLOCO_COGNITIVO = ['TIPO_DIFICULDADE_COGNITIVA', 'QTD_DIFICULDADE_COGNITIVA', 'FREQUENCIA_FALHA_MEMORIA']
SOLIDAO = ['FREQ_SOLITARIO']
COLUNAS_B = CENARIOS[nome_b]

SENSIBILIDADE = {
    'B (referência)': COLUNAS_B,
    'B sem TIPO_DIFICULDADE_COGNITIVA': COLUNAS_B + BLOCO_COGNITIVO[:1],
    'B sem bloco cognitivo': COLUNAS_B + BLOCO_COGNITIVO,
    'B sem bloco cognitivo e sem solidão': COLUNAS_B + BLOCO_COGNITIVO + SOLIDAO,
}

res_sens = {nome: treinar_avaliar(remover, **busca.best_params_) for nome, remover in SENSIBILIDADE.items()}

tabela_sens = pd.DataFrame({nome: res[0] for nome, res in res_sens.items()}).T
tabela_sens['Variação do ROC-AUC vs. B'] = tabela_sens['ROC-AUC'] - tabela_sens.loc['B (referência)', 'ROC-AUC']
tabela_sens['Nº de variáveis'] = tabela_sens['Nº de variáveis'].astype(int)
print(f'Referência de acaso: ROC-AUC 0,5 e precisão {prevalencia:.1%} (prevalência)\n')
print('Top 5 variáveis de cada cenário:')
for nome, res in res_sens.items():
    print(f'  {nome}: {", ".join(res[2].head(5).index)}')
tabela_sens.round(4)

Referência de acaso: ROC-AUC 0,5 e precisão 21.6% (prevalência)

Top 5 variáveis de cada cenário:
  B (referência): IDADE, FREQ_SOLITARIO, AUTOAVALIACAO_SAUDE_GERAL, DOR_CRONICA, VISITAS_PRONTO_ATENDIMENTO_12M
  B sem TIPO_DIFICULDADE_COGNITIVA: IDADE, FREQ_SOLITARIO, AUTOAVALIACAO_SAUDE_GERAL, QTD_DIFICULDADE_COGNITIVA, DOR_CRONICA
  B sem bloco cognitivo: IDADE, FREQ_SOLITARIO, AUTOAVALIACAO_SAUDE_GERAL, DOR_CRONICA, VISITAS_PRONTO_ATENDIMENTO_12M
  B sem bloco cognitivo e sem solidão: IDADE, AUTOAVALIACAO_SAUDE_GERAL, DOR_CRONICA, SUPORTE_SOCIAL, VISITAS_PRONTO_ATENDIMENTO_12M


,Nº de variáveis,ROC-AUC,Acurácia,Precisão (depressão),Recall (depressão),F1 (depressão),Variação do ROC-AUC vs. B
B (referência),46,0.8557,0.7995,0.5266,0.7237,0.6096,0.0000
B sem TIPO_DIFICULDADE_COGNITIVA,45,0.8545,0.7952,0.5189,0.7272,0.6057,-0.0011
B sem bloco cognitivo,43,0.8395,0.7772,0.4899,0.7302,0.5864,-0.0161
B sem bloco cognitivo e sem solidão,42,0.8126,0.7487,0.4488,0.7094,0.5498,-0.0430


### Resultado do teste de sensibilidade

| Cenário | Variáveis | ROC-AUC | Precisão | Recall | F1 |
|---|---|---|---|---|---|
| B (referência) | 46 | 0,856 | 52,7% | 72,4% | 0,610 |
| sem `TIPO_DIFICULDADE_COGNITIVA` | 45 | 0,855 | 51,9% | 72,7% | 0,606 |
| sem bloco cognitivo | 43 | 0,840 | 49,0% | 73,0% | 0,586 |
| sem bloco cognitivo e sem solidão | 42 | **0,813** | 44,9% | 70,9% | 0,550 |

- **Sem o `TIPO`:** o ROC-AUC cai só 0,001, porque `QTD_DIFICULDADE_COGNITIVA` assume o papel dela.
- **Sem o bloco cognitivo:** o ROC-AUC cai 0,016. A solidão continua entre as variáveis mais importantes.
- **Sem o bloco e sem a solidão:** o ROC-AUC é 0,813 e a precisão 44,9%, **2,1 vezes a prevalência** (21,6%). O modelo continua bem acima do acaso e o recall se mantém em 71%. As variáveis mais importantes passam a ser idade, autoavaliação de saúde, dor crônica, suporte social e visitas ao pronto-atendimento, ou seja, **saúde física e funcionamento**.
- **Mesmo padrão dos outros modelos:** a queda total vai de 0,856 para 0,813 (0,043), contra 0,856 para 0,811 no XGBoost, 0,841 para 0,804 no Random Forest e 0,816 para 0,778 no KNN.
- **Limite do teste:** ele retira só as variáveis que identificamos como suspeitas. Outras, como suporte social, também podem se relacionar com sintomas. O teste mostra robustez, não ausência total de sobreposição.

# 6. Priorizando o recall: rastreio em saúde

Em um rastreio de depressão, os dois erros não custam o mesmo. Um **falso negativo** é uma pessoa com depressão que o modelo não identifica e que, portanto, não é encaminhada para avaliação. Um **falso positivo** é uma pessoa sem depressão que passa por uma avaliação a mais. Por isso é preferível encontrar mais casos, mesmo que isso piore os falsos positivos. Esta seção mostra quanto recall dá para ganhar **baixando o limiar** de decisão do SVM e quanto isso custa: o modelo sinaliza mais pessoas, o recall sobe e a precisão cai.

**Como o limiar é escolhido, sem olhar o teste:** calculamos a distância à fronteira dos registros de treino com validação cruzada de 5 partes (`cross_val_predict`) e, para cada **recall-alvo** (80%, 85%, 90% e 95%), usamos o limiar que alcança esse recall entre os casos de depressão do treino. Depois aplicamos o limiar ao teste e medimos o que realmente aconteceu. Por isso o recall obtido no teste pode ficar um pouco acima ou abaixo do alvo.

**Como ler a tabela:** `Recall` e `Precisão` são ponderados pelo `PESO_AMOSTRAL`, como nas seções anteriores. `TP`, `FN`, `FP` e `TN` são **contagens de registros** do teste, sem peso, e `% da amostra sinalizada` e `Sinalizados por caso encontrado` também usam contagens. Este último é quantas pessoas precisam ser avaliadas para achar um caso de depressão.

In [8]:
ALVOS_RECALL = [0.80, 0.85, 0.90, 0.95]


def analisar_limiares(colunas_remover, **parametros):
    """Escolhe o limiar no treino (validação cruzada) para cada recall-alvo e mede o resultado no teste."""
    _, _, X_tr_p, X_te_p = preparar_matrizes(colunas_remover)
    escore_te = novo_svm(**parametros).fit(X_tr_p, y_tr).decision_function(X_te_p)
    # O limiar de cada recall-alvo vem dos escores de validação cruzada do TREINO; o teste só mede o recall obtido
    escore_cv = cross_val_predict(novo_svm(**parametros), X_tr_p, y_tr, cv=5,
                                  method='decision_function', n_jobs=-1)
    escore_positivos = escore_cv[(y_tr == 1).to_numpy()]

    linhas, matrizes = {}, {}
    for alvo in [None] + ALVOS_RECALL:
        limiar = 0.0 if alvo is None else np.quantile(escore_positivos, 1 - alvo)
        nome = 'Padrão (limiar 0)' if alvo is None else f'Recall-alvo {alvo:.0%}'
        metricas, matrizes[nome] = avaliar(escore_te, limiar)
        tn, fp, fn, tp = matrizes[nome].ravel()
        linhas[nome] = {
            'Limiar': limiar,
            'Recall': metricas['Recall (depressão)'],
            'Precisão': metricas['Precisão (depressão)'],
            'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
            '% da amostra sinalizada': (tp + fp) / len(y_te),
            'Sinalizados por caso encontrado': (tp + fp) / tp,
        }
    return pd.DataFrame.from_dict(linhas, orient='index'), matrizes

In [9]:
tabela_lim_b, matrizes_lim_b = analisar_limiares(COLUNAS_B, **busca.best_params_)

print(f'Cenário B (modelo ajustado). Prevalência de depressão no teste (ponderada): {prevalencia:.1%}')
print('\nMatriz de confusão com recall-alvo de 90% (nº de registros do teste; linhas = real, colunas = previsto):')
print(matrizes_lim_b['Recall-alvo 90%'])
tabela_lim_b.round(3)

Cenário B (modelo ajustado). Prevalência de depressão no teste (ponderada): 21.6%

Matriz de confusão com recall-alvo de 90% (nº de registros do teste; linhas = real, colunas = previsto):
[[2141 1341]
 [ 115  820]]


,Limiar,Recall,Precisão,TP,FN,FP,TN,% da amostra sinalizada,Sinalizados por caso encontrado
Padrão (limiar 0),0.000,0.724,0.527,688,247,658,2824,0.305,1.956
Recall-alvo 80%,-0.094,0.782,0.478,734,201,839,2643,0.356,2.143
Recall-alvo 85%,-0.206,0.829,0.439,783,152,1043,2439,0.413,2.332
Recall-alvo 90%,-0.336,0.869,0.385,820,115,1341,2141,0.489,2.635
Recall-alvo 95%,-0.543,0.944,0.329,883,52,1877,1605,0.625,3.126


### Resultado (cenário B)

- **Mais recall, mais falsos positivos.** Do limiar padrão para o recall-alvo de 90%, o recall vai de 72,4% para **86,9%** e os casos perdidos caem de 247 para **115**. O custo são os falsos positivos, que dobram (de 658 para **1.341**), e a precisão, que cai de 52,7% para 38,5%.
- **Custo em avaliações.** Para cada caso encontrado, são avaliadas 2,0 pessoas no limiar padrão, 2,6 no recall-alvo de 90% e 3,1 no de 95%.
- **Continua muito melhor que o acaso.** No recall-alvo de 90%, o modelo sinaliza 48,9% da amostra e encontra 820 dos 935 casos. Um sorteio que sinalizasse a mesma fração encontraria cerca de 460. A precisão de 38,5% é 1,8 vez a prevalência (21,6%), e mesmo no alvo de 95% ela é 1,5 vez (32,9%).
- **O recall obtido no teste fica perto do alvo.** Ele ficou entre 0,6 e 3,1 pontos abaixo dos alvos (78,2%, 82,9%, 86,9% e 94,4%), porque o limiar vem do treino.
- **Qual alvo usar é uma decisão de saúde, não estatística.** Depende do custo de avaliar cada pessoa sinalizada e da capacidade do serviço. Se a prioridade é não deixar casos para trás, o alvo de 90% é um ponto de partida razoável: cerca de 4 em cada 10 pessoas sinalizadas têm depressão, e 115 dos 935 casos (12%) ficam sem identificação.

### Mesmo teste no cenário mais conservador

Repetimos a análise sem o bloco cognitivo e sem a solidão (seção 5), as variáveis mais próximas de sintomas depressivos.

In [10]:
tabela_lim_cons, _ = analisar_limiares(COLUNAS_B + BLOCO_COGNITIVO + SOLIDAO, **busca.best_params_)
tabela_lim_cons.round(3)

,Limiar,Recall,Precisão,TP,FN,FP,TN,% da amostra sinalizada,Sinalizados por caso encontrado
Padrão (limiar 0),0.000,0.709,0.449,665,270,827,2655,0.338,2.244
Recall-alvo 80%,-0.138,0.791,0.399,738,197,1142,2340,0.426,2.547
Recall-alvo 85%,-0.245,0.847,0.366,786,149,1404,2078,0.496,2.786
Recall-alvo 90%,-0.368,0.904,0.328,843,92,1782,1700,0.594,3.114
Recall-alvo 95%,-0.542,0.950,0.279,890,45,2331,1151,0.729,3.619


- No limiar padrão, o cenário conservador encontra 665 dos 935 casos (recall de 70,9%, precisão de 44,9%). Para chegar a um recall próximo de 90% (**90,4%** no teste), o modelo sinaliza 59,4% da amostra, com 1.782 falsos positivos e precisão de **32,8%** (1,5 vez a prevalência).
- Portanto, **dá para atingir um recall alto mesmo sem as variáveis cognitivas e de solidão**, mas o custo é maior: 3,1 pessoas avaliadas por caso encontrado, contra 2,6 no cenário B completo.

# 7. O que mais dá para fazer pelo recall no SVM

Há dois tipos de alavanca para encontrar mais casos. **Deslocar o ponto de operação** (baixar o limiar, como na seção 6, ou dar mais peso à classe com depressão) aumenta o recall, mas também os falsos positivos, ao longo da mesma curva de precisão × recall. **Melhorar o modelo** sobe a curva: mais recall com os mesmos falsos positivos.

Para separar os dois, usamos duas medidas que **não dependem do limiar**: o **PR-AUC** e a **precisão com recall de 90%**, que é a maior precisão que o modelo alcança quando exigimos que ele encontre pelo menos 90% dos casos do teste. Se uma variante aumenta o recall no limiar padrão, mas deixa essas duas medidas iguais, ela só moveu o ponto de operação. Um modelo sem informação teria PR-AUC e precisão iguais à prevalência (21,6%).

Variantes testadas no cenário B, com os hiperparâmetros ajustados (`penalty='l1'`, `C = 0,1`):

1. **Peso da classe positiva maior que o `balanced`** (que equivale a cerca de 3,7): pesos 5 e 8.
2. **Sem `class_weight`**, para ver o efeito do peso sobre o ranking do modelo.
3. **Kernel RBF** (`SVC`), que permite uma fronteira não linear. Era a limitação citada na conclusão.

In [11]:
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.svm import SVC


def precisao_no_recall(escore, recall_alvo=0.90):
    """Maior precisão (ponderada) que o modelo alcança no teste exigindo ao menos `recall_alvo` de recall.

    ATENÇÃO: o limiar é escolhido olhando o próprio teste (cenário ideal). Serve só para comparar modelos e é um
    limite superior otimista; o resultado realista é o da tabela de limiares escolhidos na validação cruzada."""
    precisao, recall, _ = precision_recall_curve(y_te, escore, sample_weight=w_te)
    return precisao[recall >= recall_alvo].max()


def resumir(escore, limiar=0.0):
    """Medidas que não dependem do limiar e o resultado no limiar padrão."""
    metricas, matriz = avaliar(escore, limiar)
    tn, fp, fn, tp = matriz.ravel()
    return {
        'ROC-AUC': metricas['ROC-AUC'],
        'PR-AUC': average_precision_score(y_te, escore, sample_weight=w_te),
        'Precisão com recall de 90%': precisao_no_recall(escore),
        'Recall': metricas['Recall (depressão)'],
        'Precisão': metricas['Precisão (depressão)'],
        'Casos perdidos (FN)': fn,
        'Falsos positivos (FP)': fp,
    }

In [12]:
VARIANTES_LINEARES = {
    'Base (class_weight balanced)': novo_svm(**busca.best_params_),
    'Peso da classe positiva 5': novo_svm(class_weight={0: 1, 1: 5}, **busca.best_params_),
    'Peso da classe positiva 8': novo_svm(class_weight={0: 1, 1: 8}, **busca.best_params_),
    'Sem class_weight': novo_svm(class_weight=None, **busca.best_params_),
}
escores = {nome: modelo.fit(X_tr_b, y_tr).decision_function(X_te_b) for nome, modelo in VARIANTES_LINEARES.items()}

In [13]:
# O kernel RBF é bem mais lento que o SVM linear (cerca de 1 a 2 minutos por modelo)
for C in (0.3, 1):
    rbf = SVC(kernel='rbf', C=C, gamma='scale', class_weight='balanced')
    escores[f'Kernel RBF (C = {C})'] = rbf.fit(X_tr_b, y_tr).decision_function(X_te_b)

tabela_alavancas = pd.DataFrame.from_dict({nome: resumir(e) for nome, e in escores.items()}, orient='index')
print(f'Referência de acaso: PR-AUC e precisão com recall de 90% iguais à prevalência ({prevalencia:.1%})\n')
tabela_alavancas.round(4)

Referência de acaso: PR-AUC e precisão com recall de 90% iguais à prevalência (21.6%)



,ROC-AUC,PR-AUC,Precisão com recall de 90%,Recall,Precisão,Casos perdidos (FN),Falsos positivos (FP)
Base (class_weight balanced),0.8557,0.6717,0.3566,0.7237,0.5266,247,658
Peso da classe positiva 5,0.8545,0.6699,0.3589,0.7884,0.4737,194,862
Peso da classe positiva 8,0.8533,0.6682,0.3557,0.8474,0.4072,137,1209
Sem class_weight,0.8567,0.6728,0.3637,0.4131,0.7438,532,147
Kernel RBF (C = 0.3),0.8532,0.6577,0.3794,0.7352,0.5166,238,698
Kernel RBF (C = 1),0.8473,0.6460,0.3515,0.7033,0.5264,261,644


### Resultado

- **Dar mais peso à classe positiva aumenta o recall, mas só move o ponto de operação.** Com peso 5 e 8, o recall no limiar padrão vai de 72,4% para 78,8% e 84,7%, e os casos perdidos caem de 247 para 194 e 137. O custo são os falsos positivos, que sobem de 658 para 862 e 1.209. O ROC-AUC (0,853 a 0,857), o PR-AUC (0,668 a 0,673) e a precisão com recall de 90% (0,356 a 0,364) ficam na mesma faixa em todas as variantes lineares. É o mesmo modelo operando em outro ponto da curva, com um custo em falsos positivos praticamente igual ao de baixar o limiar (seção 6).
- **Sem `class_weight`, o ranking do modelo é o mesmo.** O ROC-AUC é 0,857 e a precisão com recall de 90% é 0,364, mas o recall cai para 41,3% (532 casos perdidos). O peso não melhora o que o modelo aprende, só muda onde ele opera por padrão.
- **O kernel RBF não ajuda.** O ROC-AUC é 0,853 (`C = 0,3`) e 0,847 (`C = 1`), abaixo do SVM linear (0,856), e o PR-AUC também cai (0,658 e 0,646). A precisão com recall de 90% de 0,379 com `C = 0,3` é a maior da tabela, mas não aparece no PR-AUC, e a diferença para a base (0,357) está dentro da variação de um único teste. Uma fronteira não linear não extrai mais informação, o que combina com o empate entre o SVM linear e o XGBoost.
- **Conclusão para o SVM:** para encontrar mais casos, o caminho é escolher o ponto de operação (o limiar da seção 6 ou um peso maior para a classe positiva). Nenhuma variante testada sobe a curva, então não há recall "de graça": cada caso a mais custa falsos positivos. O limite está na informação das variáveis, não no algoritmo.

# 8. Interpretação e conclusão

**ROC-AUC (não depende do limiar):**

| Modelo | ROC-AUC (A) | ROC-AUC (B) | B sem bloco cognitivo e sem solidão |
|---|---|---|---|
| XGBoost ajustado | 0,915 (base) | 0,856 | 0,811 |
| SVM linear ajustado | 0,915 (base) | 0,856 | 0,813 |
| KNN com colunas ponderadas | - | 0,843 | 0,796 |
| Random Forest | 0,915 | 0,841 | 0,804 |
| KNN ajustado (seção 4) | 0,833 (base, `k = 15`) | 0,816 | 0,778 |

**Pontos de operação no cenário B** (modelos ajustados). Recall e precisão são ponderados pelo `PESO_AMOSTRAL` e os falsos positivos são contagens do teste. Cada modelo tem um limiar padrão diferente. No recall-alvo de 90%, o limiar vem do treino (seção 6 deste notebook e seções equivalentes dos outros):

| Modelo | Limiar padrão: recall e precisão | Recall-alvo de 90%: recall obtido | Precisão | Falsos positivos | % da amostra sinalizada |
|---|---|---|---|---|---|
| SVM linear | 72,4% e 52,7% | 86,9% | 38,5% | 1.341 | 48,9% |
| XGBoost | 73,2% e 52,1% | 88,4% | 38,8% | 1.393 | 50,5% |
| Random Forest | 59,7% e 57,7% | 88,3% | 38,0% | 1.445 | 51,7% |
| KNN com colunas ponderadas | 66,8% e 52,9% | 88,0% | 37,0% | 1.530 | 53,7% |
| KNN ajustado (seção 4) | 45,6% e 60,8% | 87,2% | 34,0% | 1.679 | 56,8% |

**O que mais melhora o recall em cada algoritmo** (seção 7 deste notebook e seções equivalentes dos outros):

| Algoritmo | Alavanca testada | Recall no limiar padrão | Melhora a curva de precisão × recall? |
|---|---|---|---|
| Random Forest | peso 6 para a classe positiva | 59,7% → 71,5% | Não: só desloca o ponto de operação |
| KNN | colunas ponderadas pelo SVM L1 | 45,6% → 66,8% | **Sim:** PR-AUC 0,600 → 0,646 e ROC-AUC 0,816 → 0,843 |
| SVM linear | peso 5 e 8 para a classe positiva | 72,4% → 78,8% e 84,7% | Não |
| XGBoost | `scale_pos_weight` × 2 | 73,2% → 85,5% | Não |

- **A hipótese se sustenta com o SVM.** Sem nenhum sintoma de saúde mental, o modelo fica 0,36 de ROC-AUC acima do acaso (0,856) e encontra cerca de 72% dos casos de depressão, com precisão de 53% contra 21,6% de prevalência. Dados físicos, funcionais e de contexto social carregam informação real sobre depressão.
- **Como esperado, é pior que com ansiedade.** A perda é de cerca de 0,06 de ROC-AUC e de 8 pontos de recall. No cenário A, 5 das 10 variáveis mais importantes são de ansiedade.
- **O resultado é robusto.** Retirar as variáveis mais próximas de sintomas depressivos (cognição e solidão) reduz o ROC-AUC em 0,043, e o modelo continua bem acima do acaso. Sem elas, o sinal vem de idade, autoavaliação de saúde, dor crônica e suporte social.
- **Um modelo linear empata com o XGBoost.** O SVM linear (0,856) fica igual ao XGBoost ajustado (0,856) e acima do Random Forest (0,841) e do KNN (0,816, ou 0,843 com as colunas ponderadas). Isso sugere que o sinal das variáveis é em grande parte **aditivo**: modelos mais flexíveis, que capturam interações, não extraem mais informação, e o kernel RBF da seção 7 também não ajudou. Como as diferenças vêm de um único split, SVM e XGBoost são equivalentes, e o Random Forest e o KNN ponderado ficam um pouco abaixo.
- **Para o rastreio, o ponto de operação pesa mais que o modelo.** Com o recall-alvo de 90%, o SVM sinaliza 48,9% da amostra e encontra 820 dos 935 casos, deixando 115 sem identificação (247 no limiar padrão). O custo são 1.341 falsos positivos (658 no limiar padrão) e uma precisão de 38,5%, ainda 1,8 vez a prevalência. Mesmo no cenário conservador (sem cognição e sem solidão), um recall de 90% é alcançável, com precisão de 32,8%.
- **No mesmo recall, SVM, XGBoost e Random Forest se aproximam.** Com o alvo de 90%, os três sinalizam de 49% a 52% da amostra e têm precisão de 38% a 39%, uma diferença dentro da variação de um único teste. O recall mais baixo do Random Forest no limiar padrão (59,7%) vem do limiar de 0,5, não de um pior ordenamento do risco. Os dois KNN precisam de mais falsos positivos para o mesmo recall (1.530 e 1.679, contra 1.341 a 1.445).
- **Nenhuma variante do SVM sobe a curva.** O peso maior para a classe positiva (5 e 8) leva o recall no limiar padrão a 78,8% e 84,7%, mas só desloca o ponto de operação, e o kernel RBF (ROC-AUC de 0,853 e 0,847) não supera o linear. Entre os quatro algoritmos, só no KNN mudar o modelo ajudou: ponderar as colunas pelos coeficientes de um SVM L1 subiu o PR-AUC de 0,600 para 0,646.

## Limitações

- O `PESO_AMOSTRAL` entra só nas métricas, não no treino. O desbalanceamento é tratado com `class_weight='balanced'`, e o limiar padrão é a fronteira do SVM (distância zero).
- O recall-alvo (80% a 95%) é uma decisão de saúde, não estatística. O notebook mostra o custo de cada alvo, mas não define qual usar. O limiar é escolhido no treino, e o recall obtido no teste varia alguns pontos em torno do alvo.
- O kernel RBF foi testado só com `gamma = 'scale'` e dois valores de `C` (0,3 e 1), sem uma busca mais ampla. Como o SVM linear já empata com o XGBoost, não esperamos um ganho grande.
- Um único split treino/teste. A busca de hiperparâmetros usa validação cruzada, mas a comparação final usa um só teste, com cerca de 900 casos positivos. Diferenças de um a dois pontos entre variantes ficam dentro dessa variação.
- Registros completos apenas (22.084 de 24.215).
- Os modelos mostram **associação preditiva**, não causalidade.